# Appendix 10.1: Chaining Prompts

**Technique:** Prompt chaining

Feed one call's output into the next to build multistep AI workflows. Instead of cramming every task into one mega prompt, you decompose the problem into focused stages. Each call does one thing well, and its result becomes the input for the next stage.

**Contents**
* [Lesson: Why Chain?](#lesson)
* Example: Generate → Review → Fix
* Scenario: Logs → Action Items → Incident Report
* [Reflect and Stretch](#reflect)
* [Example Playground](#playground)

In [ ]:
# Make the project root importable from any working directory, then load the helpers.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lib" / "helpers.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib.helpers import get_completion, response_text, client, MODEL, MAX_TOKENS, SAMPLING_PARAMS

<a id="lesson"></a>
## Lesson: Why Chain?

A single LLM call is powerful, but it has limits:

* **Complex multipart task.** Single call: one giant prompt that is hard to debug. Chained: each stage is isolated and testable.
* **Output of step A is input to step B.** Single call: awkward to express in one prompt. Chained: natural, just pass the string forward.
* **Validation between stages.** Single call: impossible mid prompt. Chained: insert Python logic between calls.
* **Independent review pass.** Single call: model reviews its own work. Chained: separate call sees only the artefact.

### The pattern

```python
# Step 1: generate
result1 = get_completion("Write a function that does X.")

# Step 2: review the result of step 1
result2 = get_completion(f"Review this for bugs:\n{result1}")

# Step 3: apply the review back to the original
result3 = get_completion(f"""Fix the function using this review.

FUNCTION:
{result1}

REVIEW:
{result2}""")
```

Each call is a plain `get_completion(...)`. You chain them with ordinary Python variables. You can add validation, conditional branches, or loops between any two calls.

### Scenario: generate code → review it → apply fixes

A junior engineer asks Claude to write a utility function. Rather than trusting the first draft, you run a second call that reviews it for bugs and edge cases, then a third call that applies the review and returns corrected code. Three focused prompts beat one overloaded one.

In [ ]:
# Three step chain: generate → review → fix
# Each call is independent; Python variables carry the state forward.

gen = get_completion("Write a Python function `slugify(title)` that lowercases and hyphenates a title. Return only code.")
review = get_completion(f"""Review this function for bugs and edge cases. Return a bullet list.
{gen}""")
fixed = get_completion(f"""Here is a function and a review. Apply the fixes and return only the corrected code.

FUNCTION:
{gen}

REVIEW:
{review}""")

for title, text in {"GENERATED": gen, "REVIEW": review, "FIXED": fixed}.items():
    print(f"=== {title} ===\n{text}\n")

## Scenario: Logs → Action Items → Incident Report

The same pattern scales to any multistage workflow. Here the chain:

1. **Summarises** raw log lines into a concise list of observed anomalies.
2. **Extracts** concrete action items from those anomalies.
3. **Composes** a structured incident report combining the anomalies and action items.

No single prompt is asked to do all three; each stage receives exactly the context it needs.

In [ ]:
# Logs → action items → incident report
raw_logs = """2024-03-15T14:31:58Z checkout-service ERROR AttributeError in CartValidator.validate(): 'NoneType' object has no attribute 'items'
2024-03-15T14:32:01Z checkout-service ERROR HTTP 500 rate reached 42% (threshold: 5%)
2024-03-15T14:32:03Z aurora-prod-01 WARN  CPU 94%, active connections 1800/2000
2024-03-15T14:32:10Z checkout-service ERROR p99 latency 12000ms (SLO: 500ms)
2024-03-15T14:32:15Z checkout-service INFO  Deploy v2.14.1 applied 13 minutes ago"""

# Step 1: extract anomalies from logs
anomalies = get_completion(
    f"""Summarise the following service logs into a bullet list of observed anomalies. Be concise.

{raw_logs}"""
)

# Step 2: derive action items from anomalies
actions = get_completion(
    f"""Given these anomalies, list the three most important immediate action items for the on-call engineer.

{anomalies}"""
)

# Step 3: compose a structured incident report
report = get_completion(
    f"""Write a concise incident report with sections: Summary, Anomalies, Immediate Actions.

ANOMALIES:
{anomalies}

ACTIONS:
{actions}"""
)

print("=== Anomalies ===")
print(anomalies)
print("=== Action Items ===")
print(actions)
print("=== Incident Report ===")
print(report)

<a id="reflect"></a>
## Reflect and Stretch

### Reflect

**Where do you put validation between chain steps in production?**

Between any two `get_completion(...)` calls you have full Python control. Common patterns:

* **Schema validation**: parse the output with `json.loads` or a Pydantic model; raise if malformed before passing to the next step.
* **Conditional branching**: check for a keyword or confidence indicator in the output, then retry the step, escalate, or skip to a different branch.
* **Length / content guards**: if the previous step returned an empty string or an error message, abort the chain and surface a meaningful error rather than feeding garbage to the next call.
* **Rate limit backoff**: each step is a separate API call, so you can add a delay or retry between steps without touching the prompt. (The SDK already retries 429 and 5xx responses twice by default; tune it with `anthropic.Anthropic(max_retries=...)`.)

The key insight: **chaining is just Python**. The model is a function; you write ordinary control flow around it.

### Stretch challenges

1. **Add validation**: between the `gen` and `review` steps in the first cell, check that `gen` contains `def slugify`. If not, raise a `ValueError` with a helpful message before the review call fires.
2. **JSON output at the end**: add a fourth step that asks Claude to convert `fixed` into a JSON object with keys `function_name`, `code`, and `description`. Use `output_config` (Chapter 5) so it is guaranteed to parse, then `json.loads` it and print the result.
3. **Loop until clean**: wrap the review → fix cycle in a `while` loop that continues until the review returns "No issues found." (Add a max iteration guard to avoid infinite loops.)

<a id="playground"></a>
## Example Playground

Use the cell below to experiment with your own prompt chain. Swap in a different starting prompt and see how errors or nuances propagate (or get corrected) through the stages.

In [ ]:
# Build your own chain here.
# Suggested ideas:
#   * Translate docstrings to another language, then translate back to check fidelity
#   * Generate a SQL query, explain it in plain English, then suggest an index
#   * Draft a commit message, critique it, then rewrite it

step1 = get_completion("Write a one-paragraph description of what a Python generator is.")
step2 = get_completion(f"""Critique this explanation for clarity and accuracy. Return a bullet list of improvements.

{step1}""")
step3 = get_completion(f"""Rewrite the original explanation incorporating these improvements.

ORIGINAL:
{step1}

IMPROVEMENTS:
{step2}""")
print("Original:", step1)
print("\nCritique:", step2)
print("\nRevised:", step3)